# TD3 — Prétraitement EEG : des signaux connus aux signaux du dataset CL-Drive

## Objectifs pédagogiques

Ce TD est consacré au **prétraitement des signaux EEG**. Il prépare directement le projet final sur l'estimation de la charge cognitive du conducteur à partir du dataset CL-Drive.

À la fin du TD, vous devez être capables de :

1. expliquer pourquoi un signal EEG doit être prétraité ;
2. comprendre le rôle d'un filtre passe-bande ;
3. comprendre le rôle d'un filtre notch ;
4. tester une chaîne de prétraitement sur des signaux synthétiques connus ;
5. appliquer la même logique à des signaux EEG du dataset ;
6. comparer les signaux avant et après traitement dans le domaine temporel et fréquentiel ;
7. formaliser une fonction de prétraitement EEG réutilisable dans le projet.

Dans le papier CL-Drive, l'EEG est acquis avec un casque Muse à 4 canaux, échantillonné à **256 Hz**. Le prétraitement EEG indiqué par les auteurs utilise un filtre passe-bande Butterworth d'ordre 2 entre **0.4 Hz et 75 Hz**, puis un filtre notch à **60 Hz** avec facteur de qualité **30**.

## 1. Pourquoi prétraiter un signal EEG ?

Un signal EEG brut contient l'activité électrique mesurée au niveau du cuir chevelu, mais aussi des perturbations :

- dérive lente de la ligne de base ;
- bruit secteur 50 Hz ou 60 Hz ;
- bruit haute fréquence ;
- artefacts.

Le prétraitement ne sert pas à embellir le signal. Il sert à produire un signal plus exploitable pour la segmentation, l'extraction de features et la classification.

Chaîne visée :

$$
\text{EEG brut} \rightarrow \text{gestion des valeurs manquantes} \rightarrow \text{passe-bande} \rightarrow \text{notch} \rightarrow \text{EEG prétraité}
$$

### Question 1

Citer au moins trois sources d'artefacts dans un signal EEG brut.

### Réponse 

Les mouvements des yeux et clignements (artefacts oculaires),
Les mouvements musculaires comme les contractions du visage ou de la mâchoire (artefacts EMG),
Les interférences électriques du réseau électrique (par exemple bruit secteur à 50/60 Hz).

## 2. Tester le prétraitement sur des signaux connus

Avant d'appliquer une chaîne de traitement à un EEG réel, il est utile de la tester sur un signal synthétique dont on connaît exactement le contenu fréquentiel.

On construira un signal contenant :

$$
x(t) = x_{lent}(t) + x_{alpha}(t) + x_{beta}(t) + x_{secteur}(t) + b(t)
$$

avec :

- une dérive lente à 0.2 Hz ;
- une composante alpha à 10 Hz ;
- une composante beta à 25 Hz ;
- une composante de bruit secteur à 50 Hz ou 60 Hz ;
- un bruit aléatoire.

Si le traitement est correct, la dérive lente et le bruit secteur doivent être fortement réduits, tandis que les composantes utiles doivent rester visibles.

### Question 2

Pourquoi commencer par un signal synthétique avant un signal EEG réel ?

### Réponse 

Pour contrôler la dérive lente et le bruit secteur sont bien supprimés, les composantes utiles (alpha et beta) sont conservées, le traitement n’introduit pas d’erreurs ou de distorsions.

### Activité 1 — Générer un signal synthétique

Consignes méthodologiques :

1. créer un axe temporel de 20 secondes avec `fs = 256 Hz` ;
2. ajouter une sinusoïde lente à 0.2 Hz ;
3. ajouter une sinusoïde à 10 Hz ;
4. ajouter une sinusoïde à 25 Hz ;
5. ajouter une sinusoïde à 60 Hz ;
6. ajouter un bruit gaussien ;
7. tracer le signal temporel ;
8. calculer la PSD avec `scipy.signal.welch`.

Fonctions utiles : `np.arange`, `np.sin`, `np.random.normal`, `plt.plot`, `welch`.

In [ ]:
# Activité 1 — Implémenter ici la génération du signal synthétique.
# Suivre les étapes décrites dans la cellule précédente.

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import welch, butter, sosfiltfilt, filtfilt, iirnotch

fs = 256
duree = 20
t = np.arange(0, duree, 1/fs)

sin_lente = 2 * np.sin(2 * np.pi * 0.2 * t)

sin_10Hz = 1 * np.sin(2 * np.pi * 10 * t)

sin_25Hz = 0.8 * np.sin(2 * np.pi * 25 * t)

sin_60Hz = 0.5 * np.sin(2 * np.pi * 60 * t)

bruit = np.random.normal(0, 0.5, len(t))

x = sin_lente + sin_10Hz + sin_25Hz + sin_60Hz + bruit

plt.figure(figsize=(12,4))
plt.plot(t, x)
plt.xlabel("Temps (s)")
plt.ylabel("Amplitude")
plt.title("Signal EEG synthétique")
plt.xlim(0, 5)
plt.grid()
plt.show()

f, psd = welch(x, fs=fs, nperseg=1024)

plt.figure(figsize=(10,4))
plt.semilogy(f, psd)
plt.xlabel("Fréquence (Hz)")
plt.ylabel("PSD")
plt.title("Densité spectrale de puissance (Welch)")
plt.xlim(0, 80)
plt.grid()
plt.show()

## 3. Filtre passe-bande Butterworth

Un filtre passe-bande conserve les fréquences situées dans une bande choisie et atténue les autres.

Dans CL-Drive, la bande EEG retenue est :

$$
0.4 \leq f \leq 75 \quad \text{Hz}
$$

Pour un signal échantillonné à $ f_s $, la fréquence de Nyquist vaut :

$$
f_N = \frac{f_s}{2}
$$

Les fréquences de coupure doivent être normalisées :

$$
W_{low}=\frac{f_{low}}{f_N}, \qquad W_{high}=\frac{f_{high}}{f_N}
$$

Fonctions utiles :

- `scipy.signal.butter` pour concevoir le filtre ;
- `scipy.signal.sosfiltfilt` pour filtrer sans déphasage notable ;
- `scipy.signal.welch` pour vérifier l'effet fréquentiel.

### Question 3

Pourquoi utiliser `filtfilt` ou `sosfiltfilt` pour une analyse hors ligne ?

### Réponse 

On utilise filtfilt ou sosfiltfilt pour une analyse hors ligne car ces fonctions appliquent le filtre dans les deux sens, ce qui permet d’éliminer le déphasage d'un filtrage normal.

### Question 4

Calculer la fréquence de Nyquist pour `fs = 256 Hz`.

### Réponse 

$$
f_N = \frac{256}{2} = 128 Hz
$$ 

### Activité 2 — Algorithme du filtre passe-bande

Écrire une fonction de filtre passe-bande.

Algorithme :

1. fixer `lowcut = 0.4` ;
2. fixer `highcut = 75` ;
3. fixer `order = 2` ;
4. calculer `nyquist = fs / 2` ;
5. normaliser les fréquences ;
6. créer le filtre avec `butter(..., output="sos")` ;
7. appliquer le filtre avec `sosfiltfilt` ;
8. comparer signal brut et signal filtré.

In [ ]:
# Activité 2 — Implémenter ici le filtre passe-bande.

lowcut = 0.4
highcut = 75
order = 2
nyquist = fs / 2

Wlow = lowcut/nyquist
Whigh = highcut/nyquist

butter = butter(order, [Wlow, Whigh], btype='bandpass', output='sos')

filtre_sos = sosfiltfilt(butter, x)

plt.figure(figsize=(12,4))
plt.plot(t, x, label="Signal brut", alpha=0.6)
plt.plot(t, filtre_sos, label="Signal filtré", linewidth=2)
plt.xlabel("Temps (s)")
plt.ylabel("Amplitude")
plt.title("Comparaison signal brut vs signal filtré")
plt.xlim(0, 5)
plt.legend()
plt.grid()
plt.show()

# 4. Vérification fréquentielle avec la PSD
f1, psd_brut = welch(x, fs=fs, nperseg=1024)
f2, psd_filtre = welch(filtre_sos, fs=fs, nperseg=1024)

plt.figure(figsize=(10,4))
plt.semilogy(f1, psd_brut, label="Brut")
plt.semilogy(f2, psd_filtre, label="Filtré")
plt.xlabel("Fréquence (Hz)")
plt.ylabel("PSD")
plt.title("Effet du filtrage passe-bande")
plt.xlim(0, 100)
plt.legend()
plt.grid()
plt.show()

## 4. Filtre notch

Un filtre notch est contrôlé par :

- sa fréquence cible `notch_freq` ;
- son facteur de qualité `Q`.

Dans le papier, `Q = 30`.

Fonctions utiles : `scipy.signal.iirnotch`, puis `scipy.signal.filtfilt`.

### Question 5

Pourquoi CL-Drive utilise-t-il un notch à 60 Hz alors qu'en France on utilise plutôt 50 Hz ?

### Réponse 

Car l'étude est fait en Amérique du Nord où la fréquence du réseau électrique est de 60 Hz.

### Question 6

Pourquoi ne faut-il pas appliquer automatiquement un notch sans regarder la PSD ?

### Réponse 

Il ne faut pas appliquer un notch automatiquement sans regarder la PSD car on risque de supprimer une composante utile du signal au lieu d’un bruit.

### Activité 3 — Algorithme du notch

1. choisir `notch_freq` ;
2. choisir `Q = 30` ;
3. créer le filtre avec `iirnotch` ;
4. appliquer `filtfilt` ;
5. comparer les PSD avant/après ;
6. vérifier que le pic à 60 Hz diminue.

In [ ]:
# Activité 3 — Implémenter ici le filtre notch.

notch_freq = 60
Q = 30

b, a = iirnotch(notch_freq, Q, fs)
filtre = filtfilt(b, a, x)


plt.figure(figsize=(12,4))
plt.plot(t, x, label="Signal brut", alpha=0.6)
plt.plot(t, filtre, label="Signal avec notch 60 Hz", linewidth=2)
plt.xlim(0, 5)
plt.xlabel("Temps (s)")
plt.ylabel("Amplitude")
plt.title("Effet du filtre notch")
plt.legend()
plt.grid()
plt.show()

# 4. Vérification fréquentielle (PSD)
f1, psd1 = welch(x, fs=fs, nperseg=1024)
f2, psd2 = welch(filtre, fs=fs, nperseg=1024)

plt.figure(figsize=(10,4))
plt.semilogy(f1, psd1, label="Brut")
plt.semilogy(f2, psd2, label="Notch")
plt.xlim(0, 100)
plt.xlabel("Fréquence (Hz)")
plt.ylabel("PSD")
plt.title("Effet du notch filter")
plt.legend()
plt.grid()
plt.show()

## 5. Fonction complète de prétraitement EEG

La fonction finale doit enchaîner :

1. conversion en tableau numérique ;
2. gestion des valeurs manquantes courtes ;
3. filtrage passe-bande ;
4. filtrage notch ;
5. retour du signal prétraité.

Paramètres recommandés pour CL-Drive :

| Paramètre | Valeur |
|---|---:|
| `fs` | 256 Hz |
| `lowcut` | 0.4 Hz |
| `highcut` | 75 Hz |
| `order of filter` | 2 |
| `notch_freq` | 60 Hz |
| `quality_factor` | 30 |

In [ ]:
# Activité 4 — Implémenter ici une fonction preprocess_eeg_1d.
# Elle doit gérer les NaN courts, appliquer le passe-bande, puis le notch.

def preprocess_eeg_1d(x, fs=256, lowcut=0.4, highcut=75, order_of_filter=2, notch_freq=60, quality_factor=30):
    x = np.asarray(x)

    if np.isnan(x).any():
        nan_idx = np.isnan(x)
        x[nan_idx] = np.interp(np.flatnonzero(nan_idx), np.flatnonzero(~nan_idx), x[~nan_idx])  

    Wlow = lowcut/nyquist
    Whigh = highcut/nyquist

    butter = butter(order_of_filter, [Wlow, Whigh], btype='bandpass', output='sos')

    x = sosfiltfilt(butter, x)

    b, a = iirnotch(notch_freq, quality_factor, fs)
    x = filtfilt(b, a, x)

    return x

### Question 7

Quel effet attend-on du passe-bande sur la dérive lente à 0.2 Hz ?

### Réponse 

La dérive lente à 0.2 Hz est fortement atténuée voire supprimée par le filtre passe-bande.

### Question 8

Quel effet attend-on du notch sur une composante à 60 Hz ?

### Réponse 

On attend d'un filtre notch à 60Hz d'enlever les bruits du secteur.

### Question 9

Pourquoi un highcut de 75 Hz est-il valide avec fs = 256 Hz ?

### Réponse 

Avec une fréquence d’échantillonnage de fs = 256 Hz, la fréquence de Nyquist vaut : 
$$
f_N = \frac{256}{2} = 128 Hz
$$

Vu qu'un filtre passe-bande est valide tant que ses fréquences de coupure sont strictement inférieures à la fréquence de Nyquist.

highcut = 75 Hz

75 Hz < 128 Hz

### Question 10

Que se passe-t-il si `highcut > fs/2` ?

### Réponse 

Notre filtre passe-bande ne fonctionnerait pas car le filtre devient mal défini car on ne peut pas normaliser une fréquence au-dessus de Nyquist.

## 6. Passage au signal EEG multicanal

Dans CL-Drive, les canaux EEG principaux sont AF7, AF8, TP9 et TP10.

Le prétraitement doit être appliqué **canal par canal**.

Algorithme pour un DataFrame :

1. identifier la colonne temporelle si elle existe ;
2. identifier les colonnes EEG numériques ;
3. pour chaque canal : appliquer `preprocess_eeg_1d` ;
4. conserver la colonne temporelle ;
5. retourner un DataFrame filtré.

In [ ]:
# Activité 5 — Implémenter ici une fonction preprocess_eeg_dataframe.

def preprocess_eeg_dataframe(df):

    df_out = df.copy()

    possible_time_cols = ["time", "timestamp", "Timestamp", "t", "datetime"]

    time_column = next(
        (col for col in df.columns if col in possible_time_cols),
        None
    )

    eeg_cols = [
        col for col in df.columns
        if col != time_column and np.issubdtype(df[col].dtype, np.number)
    ]

    for col in eeg_cols:
        signal = df_out[col].values


        if np.isnan(signal).any():
            signal = pd.Series(signal).interpolate(limit_direction="both").values


        df_out[col] = preprocess_eeg_1d(signal)


    return df_out    

## 7. Application aux signaux EEG du dataset CL-Drive

Après validation sur signaux synthétiques, appliquer la chaîne à un fichier EEG réel :

1. choisir un fichier EEG CSV ;
2. charger avec `pd.read_csv` ;
3. identifier les colonnes EEG ;
4. afficher un extrait du signal brut ;
5. appliquer le prétraitement ;
6. afficher le même extrait après traitement ;
7. comparer la PSD avant/après.

Si le dataset n'est pas encore disponible localement, le notebook doit rester générique et indiquer clairement où définir le chemin.

### Activité 6 — Charger et visualiser un fichier EEG réel

Fonctions utiles :

| Objectif | Fonction |
|---|---|
| Chemin de fichier | `pathlib.Path` |
| Lecture CSV | `pd.read_csv` |
| Affichage des colonnes | `df.columns` |
| Tracé temporel | `plt.plot` |
| PSD | `welch` |

In [ ]:
# Activité 6 — Charger ici un fichier EEG du dataset CL-Drive.

file_path = "EEG/1030/eeg_baseline_level_1.csv"

df = pd.read_csv(file_path)

print(df.columns)  # vérification simple

df_out = preprocess_eeg_dataframe(df)

df[["AF7", "AF8", "TP9", "TP10"]].head(200).plot()
plt.title("EEG brut")
plt.show()

### Activité 7 — Comparaison avant/après sur EEG réel

Questions à traiter :

1. Le signal est-il plus exploitable après filtrage ?
2. Les variations très lentes sont-elles réduites ?
3. Observe-t-on une réduction autour de la fréquence secteur ?
4. Les canaux EEG ont-ils des comportements similaires ?
5. La PSD confirme-t-elle ce qui est visible dans le domaine temporel ?

In [ ]:
# Activité 7 — Visualiser ici les signaux EEG avant/après et leurs PSD.

### Question 11

Pourquoi faut-il comparer domaine temporel et domaine fréquentiel ?

### Réponse 

À rédiger ici.

### Question 12

Pourquoi ne faut-il pas interpoler une coupure EEG très longue ?

### Réponse 

À rédiger ici.

### Question 13

Pourquoi le prétraitement doit-il être identique pour train et test ?

### Réponse 

À rédiger ici.

## 8. Contrôle qualité après prétraitement

Vérifications minimales :

1. absence de NaN ;
2. longueur inchangée ;
3. amplitude minimale et maximale réalistes ;
4. écart-type non nul ;
5. PSD cohérente ;
6. décision d'exclure les segments trop dégradés.

In [ ]:
# Activité 8 — Implémenter un rapport qualité simple pour les canaux EEG.

## 9. Prétraitement par lot

Dans le projet final, on appliquera la même fonction à plusieurs fichiers EEG.

Algorithme :

1. définir le dossier EEG ;
2. parcourir les fichiers CSV ;
3. ignorer les fichiers déjà filtrés ;
4. charger chaque fichier ;
5. appliquer `preprocess_eeg_dataframe` ;
6. sauvegarder une copie `filtered_...csv` ;
7. produire un diagnostic.

Pour info, la structure de l'ensemble de données est la suivante :

```text
CL-Drive
    |----EEG 
         |----participant_ID_1
                      |----eeg_data_level_1
                      |----eeg_baseline_level_1
                      .
                      .
                      .
                      |----eeg_data_level_9
                      |----eeg_baseline_level_9
         .
         .
         .
         |----participant_ID_21
    |----EDA
    |----ECG
    |----Gaze
    |----Labels
```

In [ ]:
# Activité 9 — Implémenter une fonction de prétraitement par lot.
# Exemple à adapter :
# saved_files = run_batch_eeg_preprocessing(DATASET_ROOT /EEG, fs=256, notch_freq=60)